In [1]:
# make some data
import torch as t
from torch.utils.data import Dataset, DataLoader

In [3]:
class Data(Dataset):
    def __init__(self):
        self.x= t.arange(-3, 3, 0.1).view(-1,1)
        self.f = 1*self.x-1
        self.y = self.f + 0.1*t.randn(self.x.size())
        self.len = self.x.shape[0]
    # getter
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    # get length
    def __len__(self):
        return self.len

In [12]:
dataset = Data()

In [5]:
# create model and loss 

from torch import nn, optim

class LinearRegression(nn.Module):
    # constructor
    def __init__(self, input_size, output_size):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_size, output_size)

    # prediction
    def forward(self, x):
        yhat = self.linear(x)
        return yhat
    

In [6]:
criterion = nn.MSELoss()

In [8]:
model = LinearRegression(1,1)
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [9]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.1207]], requires_grad=True),
 Parameter containing:
 tensor([0.2466], requires_grad=True)]

In [10]:
optimizer.state_dict()

{'state': {},
 'param_groups': [{'lr': 0.01,
   'momentum': 0,
   'dampening': 0,
   'weight_decay': 0,
   'nesterov': False,
   'maximize': False,
   'foreach': None,
   'differentiable': False,
   'fused': None,
   'params': [0, 1]}]}

In [13]:
trainloader = DataLoader(dataset=dataset, batch_size=1)

In [14]:
model.state_dict()['linear.weight'][0]= -15
model.state_dict()['linear.bias'][0] = -10

In [15]:
list(model.parameters())

[Parameter containing:
 tensor([[-15.]], requires_grad=True),
 Parameter containing:
 tensor([-10.], requires_grad=True)]

In [16]:
# train the model: Batch Gradient Descent
def train_model_BGD(iter):
    for epoch in range(iter):
        for x,y in trainloader:
            yhat = model(x)
            loss= criterion(yhat,y)
            optimizer.zero_grad()
            loss.backward()

            optimizer.step()

train_model_BGD(10)


In [17]:
model.state_dict()

OrderedDict([('linear.weight', tensor([[1.0009]])),
             ('linear.bias', tensor([-0.9857]))])

# Training and validation 

In [1]:
import torch as t
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import numpy as np

In [3]:
# make some data
class Data(Dataset):
    def __init__(self, train= True):
        self.x = t.arange(-3, 3).view(-1,1)
        self.f = -3*self.x -1
        self.y = self.f + 0.1*t.randn(self.x.size())
        self.len = self.x.shape[0]
        if train == True:
            self.y[0]= 0
            self.y[50:55]= 20
        else:
            pass

    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return self.len

In [19]:
train_data =Data()
validation_data = Data(train=False)

In [16]:
#  create linear regression data loader and criterion


class LinearRegression(nn.Module):
    def __init__(self, input_size, output_size):
        super(LinearRegression,self).__init__()
        self.linear = nn.Linear(input_size, output_size)
    # prediction
    def forward(self, x):
        yhat = self.linear(x)
        return yhat
    

In [6]:
criterion = nn.MSELoss()

In [7]:
trainloader = DataLoader(dataset=train_data, batch_size=1)

In [11]:
# learning rates and data structures to store results of different hyperparameters
learning_rates = [0.0001, 0.001, 0.01, 0.1]
train_error = t.zeros(len(learning_rates))
validation_error = t.zeros(len(learning_rates))
MODELS = []

In [20]:
# try different values of learning rates , perform SDG, store the results in training and validation data and save the results on MOdels list
def train_model_with_lr(iter, lr_list):
    for i, lr in enumerate(lr_list):
        model = LinearRegression(1,1)
        optimizer = optim.SGD(model.parameters(), lr=lr)
        for epoch in range(iter):
            for x,y in trainloader:
                yhat = model(x)
                loss = criterion(yhat, y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
        # train data
        Yhat = model(train_data.x)
        train_loss = criterion(Yhat, train_data.y)
        train_error[i]= train_loss.item()

        # validation data 
        Yhat = model(validation_data.x)
        valid_loss = criterion(Yhat, validation_data.y)
        validation_error[i] = valid_loss.item()
        MODELS.append(model)


In [21]:
train_model_with_lr(10, learning_rates)

RuntimeError: mat1 and mat2 must have the same dtype, but got Long and Float

# Multiple Linear Regression

In [1]:
import torch as t
from torch import nn
t.manual_seed(42)

In [2]:
# weight and bias
w = t.tensor([[2.0], [3.0]], requires_grad=True)
b = t.tensor([[1.0]])

In [3]:
# prediction
def forward(x):
    yhat = t.mm(x,w) +b
    return yhat

In [6]:
x = t.tensor([[1.0, 2.0]])

In [8]:
yhat= forward(x)
print(f'Prediction: {yhat}')

Prediction: tensor([[9.]], grad_fn=<AddBackward0>)


In [9]:
X = t.tensor([[1.0, 1.0], [1.0, 2.0], [2.0, 3.0]])

In [10]:
Yhat = forward(X)
Yhat

tensor([[ 6.],
        [ 9.],
        [14.]], grad_fn=<AddBackward0>)

In [11]:
model = nn.Linear(2,1)

In [14]:
yhat_model = model(x)
yhat_model

tensor([[1.5488]], grad_fn=<AddmmBackward0>)

In [16]:
list(model.parameters())

[Parameter containing:
 tensor([[0.5406, 0.5869]], requires_grad=True),
 Parameter containing:
 tensor([-0.1657], requires_grad=True)]

In [17]:
Yhat_model = model(X)
Yhat_model

tensor([[0.9619],
        [1.5488],
        [2.6763]], grad_fn=<AddmmBackward0>)

In [18]:
model.state_dict()

OrderedDict([('weight', tensor([[0.5406, 0.5869]])),
             ('bias', tensor([-0.1657]))])

In [19]:
# Custom Modules

class LinearRegression(nn.Module):
    def __init__(self, input_size, output_size):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_size, output_size)
    
    def forward(self, x):
        yhat = self.linear(x)
        return yhat

In [20]:
custom_model = LinearRegression(2,1)

In [22]:
list(custom_model.parameters())

[Parameter containing:
 tensor([[ 0.6496, -0.1549]], requires_grad=True),
 Parameter containing:
 tensor([0.1427], requires_grad=True)]

In [23]:
custom_model.state_dict()

OrderedDict([('linear.weight', tensor([[ 0.6496, -0.1549]])),
             ('linear.bias', tensor([0.1427]))])

In [24]:
yhat = custom_model(x)
print(f'Custom model prediction: {yhat}')

Custom model prediction: tensor([[0.4824]], grad_fn=<AddmmBackward0>)


In [25]:
Yhat = custom_model(X)

print(f'Prediction{Yhat}')

Predictiontensor([[0.6373],
        [0.4824],
        [0.9770]], grad_fn=<AddmmBackward0>)


# Training for MLR


In [37]:
from torch import optim, nn
from torch.utils.data import Dataset, DataLoader

In [47]:
# make some data
class Data(Dataset):
    def __init__(self):
        self.x = t.zeros(20,2)
        self.x[:,0] = t.arange(-1,1, 0.1)
        self.x[:,1] = t.arange(-1, 1, 0.1)
        self.w = t.tensor([[1.0], [2.0]])
        self.b = t.tensor([[1.0]])
        self.f = t.mm(self.x, self.w)+ self.b
        self.y = self.f + 0.1*t.randn(self.x.shape[0],1)
        self.len = self.x.shape[0]
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

In [48]:
# dataset object
data_set = Data()

In [49]:
# model
class LinearRegression(nn.Module):
    def __init__(self, input_size, output_size):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_size, output_size)
    
    # prediction
    def forward(self, x):
        yhat = self.linear(x)
        return yhat

In [50]:
model = LinearRegression(2,1)

In [52]:
# optimizer
optimizer = optim.SGD(model.parameters(), lr = 0.1)

In [53]:
criterion = nn.MSELoss()

In [54]:
train_loader = DataLoader(dataset = data_set, batch_size = 2)

In [55]:
# training
LOSS= []
def train_model(epochs):
    for e in range(epochs):
        for x, y in train_loader:
            yhat = model(x)
            loss = criterion(yhat, y)
            LOSS.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [56]:
train_model(100)